# YouTube Transcript RAG System

A Retrieval-Augmented Generation (RAG) system built with LangChain that answers questions about a **YouTube video** using its transcript as the knowledge source.

This notebook is designed to be run **cell by cell** and demonstrated live in a GenAI / RAG interview.

## What this system does

```
YouTube video URL  ->  Transcript  ->  Cleaning  ->  Chunking
                                                        |
                                                        v
                                               Embeddings + FAISS
                                                        |
                                                        v
User question  ->  Query embedding  ->  Similarity search  ->  Top-k chunks
                                                                     |
                                                                     v
                                                        Prompt + Context -> Gemini
                                                                     |
                                                                     v
                                                          Grounded natural-language answer

```

## Why this design

- **No external login or premium account is required.** The only credential required is a `GOOGLE_API_KEY`.
- **The transcript is the only knowledge source.** This keeps the demo simple, fast, and easy to reason about.
- **Every RAG stage is visible**: ingestion, chunking, embeddings, vector storage, retrieval, and generation are each their own section so you can point at exactly what is happening at each step.

## Architecture

```
                   USER
                    |
                    v
             YouTube URL
                    |
                    v
          Transcript Extraction
                    |
                    v
            Text Preprocessing
                    |
                    v
                Chunking
                    |
                    v
              Embeddings
                    |
                    v
                 FAISS
              Vector Store
                    |
                    |
             User Question
                    |
                    v
             Query Embedding
                    |
                    v
          Similarity Retrieval
                    |
                    v
            Top-K Chunks
                    |
                    v
            Prompt + Context
                    |
                    v
                 Gemini
                    |
                    v
             Grounded Answer
```

### RAG = Retrieval + Augmentation + Generation

- **Retrieval** - FAISS finds the transcript chunks whose embeddings are closest (most semantically similar) to the question's embedding.
- **Augmentation** - Those retrieved chunks are inserted into the LLM prompt as `{context}`, alongside the user's `{question}`.
- **Generation** - Gemini reads the question _and_ the retrieved context, then generates a natural-language answer grounded in that context (instead of relying purely on what it memorized during training).

### Why we don't just ask the LLM the question directly

The LLM was never trained on this specific video's transcript, and even if it had general knowledge of the topic, it has no way to know what was _actually said_ in _this_ video. RAG solves this by retrieving the relevant transcript text at query time and grounding the answer in it.


## 1. Install Dependencies

Only the packages needed for this YouTube transcript RAG pipeline are installed:

- `youtube-transcript-api` - retrieves captions/transcripts without downloading the video
- `langchain`, `langchain-community`, `langchain-google-genai`, `langchain-text-splitters` - RAG orchestration, embeddings, and chunking
- `faiss-cpu` - vector similarity search
- `python-dotenv` - loads the `GOOGLE_API_KEY` environment variable

In [ ]:
%pip install --upgrade youtube-transcript-api python-dotenv langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu

## 2. Imports & Configuration

The only required credential is `GOOGLE_API_KEY`, used for Gemini and Google embeddings. Put it in a `.env` file in the same folder as this notebook:

```
GOOGLE_API_KEY=your_key_here
```

The pipeline does not require a YouTube API key or video download.

In [ ]:
import os
import re
from urllib.parse import parse_qs, urlparse

from dotenv import load_dotenv

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    CouldNotRetrieveTranscript,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

In [ ]:
# Verify API key
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError(
        "GOOGLE_API_KEY is not configured. "
        "Create a .env file with GOOGLE_API_KEY=your_key_here and re-run this cell."
    )

print("GOOGLE_API_KEY loaded successfully.")

## 3. YouTube Transcript Ingestion

This section retrieves the transcript and lightweight metadata for a YouTube video:

1. Parse a YouTube URL to extract the video ID.
2. Fetch manually created captions, falling back to auto-generated captions.
3. Stitch transcript segments into readable text.

No video is downloaded, which keeps the pipeline lightweight and fast enough to demo live.

Supported URL formats:

```
https://www.youtube.com/watch?v=VIDEO_ID
https://youtu.be/VIDEO_ID
https://www.youtube.com/embed/VIDEO_ID
https://www.youtube.com/shorts/VIDEO_ID
```

In [ ]:
def extract_video_id(video_url: str) -> str:
    """Extract an 11-character YouTube video ID from common URL formats."""
    if not isinstance(video_url, str) or not video_url.strip():
        raise ValueError("Could not extract a valid YouTube video ID.")

    parsed = urlparse(video_url.strip())
    hostname = parsed.netloc.lower().split(":")[0]
    video_id = ""

    if hostname in {"youtu.be", "www.youtu.be"}:
        video_id = parsed.path.lstrip("/").split("/")[0]
    elif hostname in {"youtube.com", "www.youtube.com", "m.youtube.com"}:
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [""])[0]
        elif parsed.path.startswith(("/embed/", "/shorts/", "/live/")):
            video_id = parsed.path.split("/")[2]

    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError("Could not extract a valid YouTube video ID.")
    return video_id

In [ ]:
def get_youtube_transcript(video_url: str) -> dict:
    """Return transcript text and lightweight metadata without a YouTube API key."""
    result = {
        "video_id": None,
        "source": video_url,
        "title": None,
        "transcript": "",
        "error": None,
    }

    try:
        video_id = extract_video_id(video_url)
    except ValueError as error:
        result["error"] = str(error)
        return result

    result["video_id"] = video_id
    result["source"] = f"https://www.youtube.com/watch?v={video_id}"
    result["title"] = f"YouTube video ({video_id})"

    try:
        api = YouTubeTranscriptApi()
        transcript_list = api.list(video_id)
        try:
            transcript = transcript_list.find_manually_created_transcript(["en"])
        except NoTranscriptFound:
            transcript = transcript_list.find_generated_transcript(["en"])
        fetched = transcript.fetch()
        text = " ".join(segment.text.strip() for segment in fetched if segment.text.strip())
        result["transcript"] = re.sub(r"\s+", " ", text).strip()
        if not result["transcript"]:
            result["error"] = "No transcript could be retrieved for this video."
    except (TranscriptsDisabled, NoTranscriptFound, CouldNotRetrieveTranscript, VideoUnavailable) as error:
        result["error"] = (
            "No transcript could be retrieved for this video. "
            "Try another video with captions/transcripts enabled."
        )
    except Exception as error:
        result["error"] = f"Unexpected error while fetching transcript: {error}"

    return result

### Optional ingestion check

To avoid an unsolicited network call, run the next cell only after replacing the URL with a video that has captions. It verifies transcript extraction before you build embeddings.

In [ ]:
# Replace this with a known video URL before running the check.
DEMO_VIDEO_URL = ""

if not DEMO_VIDEO_URL:
    print("Set DEMO_VIDEO_URL to a captioned YouTube URL, then run this cell.")
else:
    demo_result = get_youtube_transcript(DEMO_VIDEO_URL)
    if demo_result["error"]:
        print("Error:", demo_result["error"])
    else:
        print("Video ID:", demo_result["video_id"])
        print("Title:", demo_result["title"])
        print("Transcript length (chars):", len(demo_result["transcript"]))
        print("Preview:", demo_result["transcript"][:300], "...")

## 4. Document Processing (Chunking)

We now turn the raw transcript into LangChain `Document` objects, split into overlapping chunks.

### Why chunking is necessary

We cannot send the entire transcript to the LLM every time - long videos can produce transcripts far larger than a single prompt should reasonably contain (cost, latency, and the "lost in the middle" problem where LLMs pay less attention to the middle of very long contexts). Instead, we split the transcript into smaller, semantically searchable **chunks**, embed each chunk, and at query time retrieve only the chunks that are actually relevant to the question. This is the "R" (Retrieval) in RAG.

- `chunk_size=1000`: each chunk is roughly 1000 characters - large enough to preserve context, small enough to stay focused on one topic.
- `chunk_overlap=200`: consecutive chunks share 200 characters so a sentence that gets cut at a chunk boundary is not stranded without context.

Every chunk keeps the parent video's metadata (`video_id`, `source`, `title`) so we can always trace an answer back to its source video.


In [ ]:
def create_documents(transcript_results) -> list:
    """
    Turn one or more transcript results (as returned by get_youtube_transcript)
    into chunked LangChain Document objects, ready for embedding.

    Accepts either a single transcript result dict or a list of them,
    which is what enables the optional multi-video mode later in this notebook.
    """
    if isinstance(transcript_results, dict):
        transcript_results = [transcript_results]

    documents = []
    for result in transcript_results:
        if result.get("error") or not result.get("transcript"):
            print(
                f"Skipping {result.get('source')}: {result.get('error') or 'empty transcript'}"
            )
            continue

        documents.append(
            Document(
                page_content=result["transcript"],
                metadata={
                    "video_id": result["video_id"],
                    "source": result["source"],
                    "title": result["title"],
                },
            )
        )

    if not documents:
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)

    return split_docs

## 5. Chunk Inspection

Before building the vector store, it's worth looking at what the chunking step actually produced. This is useful to show during a live demo - it makes the ingestion pipeline concrete rather than abstract.


In [ ]:
def inspect_chunks(chunks, preview_count: int = 3):
    """Print a quick summary of the chunking output for debugging/demo purposes."""
    print(f"Number of chunks: {len(chunks)}")
    if not chunks:
        return

    lengths = [len(c.page_content) for c in chunks]
    print(f"Average chunk length (chars): {sum(lengths) // len(lengths)}")
    print(f"Min / Max chunk length (chars): {min(lengths)} / {max(lengths)}")
    print("-" * 60)

    for i, chunk in enumerate(chunks[:preview_count], start=1):
        print(f"Chunk {i}:")
        print(
            chunk.page_content[:400] + ("..." if len(chunk.page_content) > 400 else "")
        )
        print("Metadata:", chunk.metadata)
        print("-" * 60)

## 6. Embeddings + FAISS Vector Store

### Why a vector database?

Keyword matching alone can miss content that is semantically related but doesn't share exact words. For example, the question:

> "What did the speaker say about job displacement?"

could - and should - retrieve a transcript chunk containing something like:

> "AI may replace certain repetitive tasks..."

even though the words "job displacement" never appear in that chunk. Embeddings capture _meaning_, not just keywords, so a vector similarity search can surface this chunk anyway.

### The flow

```
Transcript chunks
      |
      v
Embedding model (Google Generative AI)
      |
      v
Vector representations (high-dimensional numeric vectors)
      |
      v
FAISS (stores vectors, supports fast similarity search)
```


In [ ]:
def build_vectorstore(documents):
    """
    Embed transcript chunks with Google's current embedding model and store
    the vectors in FAISS for fast semantic similarity search.
    """
    if not documents:
        raise ValueError(
            "No documents were provided - build the transcript chunks first."
        )

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=google_api_key,
    )
    return FAISS.from_documents(documents, embeddings)

## 7. Retriever

`k` controls how many relevant chunks are retrieved and passed to the LLM. A larger `k` can provide more context but may introduce irrelevant information and increase token usage/cost. A smaller `k` is cheaper and more focused but risks missing relevant context - this trade-off is worth calling out explicitly in an interview.


In [ ]:
def get_retriever(vectorstore, k: int = 4):
    """Return a configured retriever. k is intentionally explicit and easy to tune."""
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

## 8. Grounded Prompt + RAG Chain

The system prompt explicitly instructs Gemini to:

1. Answer using **only** the retrieved transcript context.
2. Say so explicitly if the answer cannot be determined from the context - never invent facts.
3. Distinguish between what the transcript states directly and any reasonable interpretation.
4. Stay concise.
5. Reference the source video when useful.


In [ ]:
RAG_SYSTEM_PROMPT = """You are a grounded YouTube transcript assistant.

Answer the user's question using ONLY the retrieved transcript context below.

Rules:
1. Do not invent facts that are not supported by the context.
2. If the answer cannot be determined from the context, explicitly say the transcript does not contain enough information to answer - do not guess.
3. Base your answer primarily on the retrieved transcript, not on outside knowledge.
4. If you go beyond what is literally stated (reasonable interpretation), say so clearly.
5. Keep the answer concise but complete.
6. Mention the relevant video/source when it is useful to the user.

Retrieved context:
{context}
"""


def format_docs(docs) -> str:
    """Join retrieved chunks into a single context string, tagged by source."""
    formatted = []
    for index, doc in enumerate(docs, start=1):
        title = doc.metadata.get("title", "Unknown source")
        formatted.append(f"[{index}] (Source: {title})\n{doc.page_content}")
    return "\n\n".join(formatted)


def build_rag_chain(vectorstore, k: int = 4):
    """Wire the configured retriever, grounded prompt, Gemini, and output parser."""
    retriever = get_retriever(vectorstore, k=k)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2,
        google_api_key=google_api_key,
    )
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", RAG_SYSTEM_PROMPT),
            ("human", "{question}"),
        ]
    )
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain, retriever

## 9. Query Processing: Show Retrieval + Generation Together

This is the most important part to demonstrate live: the notebook does **not** just print a final answer. It shows the retrieved chunks (with similarity scores) _before_ showing the generated answer, so the RAG process is visually obvious.

Note on scores: `similarity_search_with_score` in FAISS (via LangChain) returns an L2 distance by default - **lower is more similar**. This is a similarity/distance score, not a measure of the LLM's confidence in its answer, and it is labeled as such below.


In [ ]:
def retrieve_context(vectorstore, question: str, k: int = 4):
    """Return (document, FAISS L2 distance) pairs; lower distance is more similar."""
    if not question or not question.strip():
        raise ValueError("Please provide a question.")
    if k < 1:
        raise ValueError("k must be at least 1.")
    return vectorstore.similarity_search_with_score(question, k=k)


def ask_question(question: str, vectorstore=None, rag_chain=None, k: int = 4):
    """Retrieve, display evidence, generate a grounded answer, and show sources."""
    if vectorstore is None:
        return "Please build the vector store first using build_vectorstore()."
    if rag_chain is None:
        return "Please build the RAG chain first using build_rag_chain()."

    print("=" * 70)
    print("QUESTION")
    print(question)
    print("=" * 70)

    scored_docs = retrieve_context(vectorstore, question, k=k)
    print("\nRETRIEVED CONTEXT (FAISS L2 distance - lower = more similar)")
    print("-" * 70)
    for index, (doc, score) in enumerate(scored_docs, start=1):
        title = doc.metadata.get("title", "Unknown source")
        preview = doc.page_content[:300] + ("..." if len(doc.page_content) > 300 else "")
        print(f"{index}. [distance={score:.4f}] (Source: {title})")
        print(f'   "{preview}"\n')

    answer = rag_chain.invoke(question)
    print("=" * 70)
    print("GENERATED ANSWER")
    print("=" * 70)
    print(answer)

    print("\n" + "=" * 70)
    print("SOURCE(S)")
    print("=" * 70)
    seen_sources = set()
    for doc, _ in scored_docs:
        source = doc.metadata.get("source", "Unknown source")
        if source not in seen_sources:
            print(f"- {doc.metadata.get('title', 'Unknown source')}: {source}")
            seen_sources.add(source)

    return answer

## 10. Interactive Interface

Provide a YouTube URL and a question. If you don't have one handy, a reliable demo video (3Blue1Brown's "But what is a neural network?", which has clean English captions) is used as a fallback.


In [ ]:
VIDEO_URL = input("Enter YouTube URL: ").strip()
QUESTION = input("Ask a question about the video: ").strip()

if not VIDEO_URL:
    print("Please paste a YouTube URL before running the end-to-end demo.")
if not QUESTION:
    print("Please enter a question about the video before running the end-to-end demo.")

## 11. End-to-End Demo

Runs the full pipeline: transcript extraction -> chunking -> embeddings -> FAISS -> retrieval -> Gemini -> grounded answer.


In [ ]:
# Step 1: Get the transcript
if not VIDEO_URL or not QUESTION:
    print("Enter both a YouTube URL and a question in the previous cell.")
else:
    transcript_result = get_youtube_transcript(VIDEO_URL)

    if transcript_result["error"]:
        print("Error:", transcript_result["error"])
    else:
        # Step 2: Chunk the transcript into metadata-rich Documents
        chunks = create_documents(transcript_result)
        inspect_chunks(chunks)

        # Step 3: Build the FAISS vector store from Google embeddings
        vectorstore = build_vectorstore(chunks)

        # Step 4: Build the grounded prompt + Gemini chain
        K = 4
        rag_chain, retriever = build_rag_chain(vectorstore, k=K)

        # Step 5: Retrieve, display evidence, and generate the answer
        answer = ask_question(QUESTION, vectorstore=vectorstore, rag_chain=rag_chain, k=K)

## 12. RAG Evaluation Tests

A few predefined questions to sanity-check the system's behavior. Useful to run live in an interview to demonstrate that the system is genuinely grounded rather than just calling an LLM.

Run the end-to-end demo cell above first so `vectorstore` and `rag_chain` exist. Update the questions below to match whichever video you loaded.

- **Test 1 - Direct fact**: a question explicitly answered in the transcript. Expect a correct, grounded answer.
- **Test 2 - Semantic retrieval**: a question phrased very differently from the transcript's wording, to confirm retrieval works on meaning, not just keyword overlap.
- **Test 3 - Out-of-context**: a question the video does not address. Expect the model to say the transcript doesn't contain enough information, rather than hallucinating.
- **Test 4 - Multi-part**: a question that likely needs information spread across more than one chunk, to demonstrate why retrieving multiple chunks (k > 1) matters.


In [ ]:
# Update these questions to fit whichever video you loaded above.
test_questions = [
    "Direct fact: What specific example or term does the speaker introduce early in the video?",
    "Semantic retrieval: In different words than the transcript uses, what problem is this video trying to explain or solve?",
    "Out-of-context: What does the speaker say about the stock market?",
    "Multi-part: What are two distinct ideas or steps the speaker covers, and how do they relate to each other?",
]

if "vectorstore" not in globals() or "rag_chain" not in globals():
    print("Run the end-to-end demo successfully before running evaluation tests.")
else:
    for question in test_questions:
        ask_question(question, vectorstore=vectorstore, rag_chain=rag_chain, k=K)
        print("\n")

## 13. Optional: Multi-Video RAG

This section is optional and not required for the primary single-video demo above. It shows how the same pipeline extends to a **knowledge base across multiple videos** - useful for comparison questions like "How do these two videos differ in their explanation of X?"

The only change is that `create_documents` (already written above) accepts a list of transcript results and tags every chunk with its own video's metadata, so retrieval can surface chunks from either video for a single question.


In [ ]:
# Example (optional) - uncomment and fill in real URLs to try multi-video retrieval
# VIDEO_URLS = [
#     "https://www.youtube.com/watch?v=VIDEO_ID_1",
#     "https://www.youtube.com/watch?v=VIDEO_ID_2",
# ]
#
# transcript_results = [get_youtube_transcript(url) for url in VIDEO_URLS]
# multi_chunks = create_documents(transcript_results)
# inspect_chunks(multi_chunks)
#
# multi_vectorstore = build_vectorstore(multi_chunks)
# multi_rag_chain, multi_retriever = build_rag_chain(multi_vectorstore, k=4)
#
# ask_question(
#     "How do these videos differ in their explanation of the topic?",
#     vectorstore=multi_vectorstore,
#     rag_chain=multi_rag_chain,
#     k=4,
# )

## 14. Interview Talking Points

- **Why RAG instead of just prompting the LLM directly?** The LLM has no knowledge of this specific video's content. RAG grounds the answer in retrieved transcript text, reducing hallucination and making answers verifiable against a source.
- **Why chunk instead of sending the whole transcript?** Cost, latency, and context-window limits - plus retrieval lets us send only the _relevant_ parts of a potentially long transcript.
- **Why embeddings instead of keyword search?** Embeddings capture semantic meaning, so a question can retrieve relevant content even when it doesn't share exact wording with the transcript.
- **What does `k` control, and what's the trade-off?** How many chunks are retrieved and passed to the LLM - more context vs. more noise/cost.
- **How do we know the system isn't hallucinating?** The system prompt explicitly instructs the model to say so when the context is insufficient, and the out-of-context test above verifies that behavior.
- **How would this scale?** Swap FAISS for a managed vector database (e.g. Pinecone, Chroma, pgvector) and the multi-video pattern shown above already demonstrates how the knowledge base would grow to many sources.
